<div dir="rtl" align="right">

# تحليلُ الإنتروبيا لِقياسِ تعقيدِ الإشارةِ

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

تَقيسُ الإنتروبيا درجةَ التعقيدِ أو عدمِ اليقينِ في الإشارة. نَحسبُ ثلاثةَ أنواعٍ: إنتروبيا العيّنةِ، والإنتروبيا التقريبيّة، والإنتروبيا الطيفيّة، على نوافذَ مُتحرّكةٍ من القناةِ P4.

## المُخرجاتُ المُتوقّعةُ

- ثلاثةُ مخططاتٍ: الإشارةُ الأصليّة، إنتروبيا العيّنةِ والتقريبيّة، الإنتروبيا الطيفيّة
- تَذبذبٌ في قيمِ الإنتروبيا يَعكسُ تَغيّرَ تعقيدِ الإشارةِ عبرَ الزمن

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| WINDOW | 1000 | نافذةُ 5 ثوانٍ |
| STEP | 500 | خطوةُ 2.5 ثانيةٍ |
| order | 2 | بُعدُ التضمينِ |
| tolerance | 0.2*std | عتبةُ التشابهِ |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly antropy mne wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2.

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ تحليلِ الإنتروبيا

نَحسبُ ثلاثةَ مقاييسَ للإنتروبيا على نوافذَ مُتحرّكةٍ من القناةِ P4.

</div>

In [ ]:
from antropy import sample_entropy, app_entropy, spectral_entropy

WINDOW = 1000
STEP = 500
channel_data = eeg_data[:, 0]

n_windows = (len(channel_data) - WINDOW) // STEP + 1
sampen_vals = []
appen_vals = []
specen_vals = []
window_centers = []

for i in range(n_windows):
    start = i * STEP
    end = start + WINDOW
    segment = np.ascontiguousarray(channel_data[start:end])
    r_val = 0.2 * np.std(segment)
    sampen_vals.append(sample_entropy(segment, order=2, tolerance=r_val))
    appen_vals.append(app_entropy(segment, order=2, tolerance=r_val))
    specen_vals.append(spectral_entropy(segment, sf=fs, method='welch', normalize=True))
    window_centers.append((start + end) / 2 / fs)

print(f'Computed {n_windows} windows')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- المناطقُ ذاتُ التعقيدِ الأعلى تُظهرُ إنتروبيا أعلى
- إنتروبيا العيّنةِ والتقريبيّةِ تَتتبّعانِ بعضَهما بِشكلٍ عامٍّ
- الإنتروبيا الطيفيّةُ تَقيسُ توزيعَ الطاقةِ على الترددات


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sig = np.arange(n_plot) / fs

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Original signal - Channel P4',
                                    'Sample and Approximate Entropy',
                                    'Spectral Entropy (normalized)'))
fig.add_trace(go.Scatter(x=t_sig, y=channel_data[:n_plot], name='Signal',
                         line=dict(color='blue', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=window_centers, y=sampen_vals, name='Sample Entropy',
                         line=dict(color='green', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=window_centers, y=appen_vals, name='Approximate Entropy',
                         line=dict(color='orange', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=window_centers, y=specen_vals, name='Spectral Entropy',
                         line=dict(color='purple', width=1.5)), row=3, col=1)
fig.update_layout(height=900, title_text='Entropy Analysis - Channel P4',
                  xaxis3_title='Time (s)', showlegend=True)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- الإنتروبيا تَقيسُ تعقيدَ الإشارةِ دونَ تعلّمِ الآلة
- إنتروبيا العيّنةِ أداةٌ شائعةٌ لِمراقبةِ الإجهادِ الذهنيّ
- الإنتروبيا الطيفيّةُ تَكشفُ توزيعَ الطاقةِ على الترددات
- النوافذُ المُتحرّكةُ تَكشفُ التغيّراتِ الزمنيّةَ في التعقيد


</div>